# Dark Manifold Virtual Cell — validation against Breuer 2019

Loads the trained `dark_manifold_100gene.pt` checkpoint from `Nikku03/cell-simulation` and validates it against the JCVI-Syn3A essentiality labels in `Nikku03/cell` (Breuer et al. 2019, eLife).

Closes the validation gap explicitly named in `cell-simulation/GAP_ANALYSIS.md`: **"No validation against experimental essentiality data."**

Method:
1. Clone both repos.
2. Load `DarkManifold100Gene` checkpoint.
3. For each of the 102 genes the model knows, run a knockout via `enzyme_mask[gene_idx] = 0`, roll out a trajectory, derive an essentiality score from the predicted ATP / energy-charge drop.
4. Map gene names (`pyk`, `rpoA`, ...) → Syn3A locus tags (`JCVISYN3A_xxxx`) via the GenBank `/gene` qualifier.
5. Join with the 455 Breuer 2019 essentiality labels in `labels.csv`. Compute MCC at the optimal threshold and report.

Realistic expectation per the existing `GAP_ANALYSIS.md`: modest MCC (~0.2–0.4). Hub-gene wrong-sign errors and FBA-synthetic training data bound the ceiling. The number itself is the deliverable — a **measured** validation against real experimental labels.

Runs on Colab (CPU is enough; the model is ~1 MB).

---

## Cell 1 — clone repos + install deps

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Clone Dark Manifold model repo
DM_REPO = Path('/content/cell-simulation')
if not DM_REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Nikku03/cell-simulation.git',
                    str(DM_REPO)], check=True)
print(f'Dark Manifold repo: {DM_REPO}')

# Clone the Syn3A simulator repo (for Breuer labels)
CELL_REPO = Path('/content/cell')
if not CELL_REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    '-b', 'claude/syn3a-whole-cell-simulator-REjHC',
                    'https://github.com/Nikku03/cell.git',
                    str(CELL_REPO)], check=True)
print(f'Cell repo: {CELL_REPO}')

# Clone the Luthey-Schulten Minimal_Cell_ComplexFormation repo INTO the cell
# tree so the existing path cell_sim/data/Minimal_Cell_ComplexFormation/...
# resolves. The cell repo intentionally does NOT commit this third-party
# data — users clone it separately via cell_sim/setup.sh in normal use.
MCCF = CELL_REPO / 'cell_sim/data/Minimal_Cell_ComplexFormation'
if not (MCCF / 'input_data/syn3A.gb').exists():
    # Remove any empty placeholder dir, then clone
    if MCCF.exists():
        import shutil; shutil.rmtree(MCCF)
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/Luthey-Schulten-Lab/Minimal_Cell_ComplexFormation.git',
                    str(MCCF)], check=True)
assert (MCCF / 'input_data/syn3A.gb').exists(), 'Syn3A GenBank still missing'
print(f'Luthey-Schulten data: {MCCF}')

# Install deps (torch should already be on Colab; biopython for GenBank)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'biopython', 'pandas', 'numpy', 'scikit-learn'],
               check=True)

sys.path.insert(0, str(DM_REPO))
print('OK')

## Cell 2 — load the Dark Manifold checkpoint

The checkpoint is a dict with `model_state_dict`, `genes` (list of 102 names), `metabolites` (list of 74), and `results` (already-computed quick-eval results).

In [ ]:
import torch, json, numpy as np

ckpt = torch.load(DM_REPO / 'dark_manifold_100gene.pt',
                  map_location='cpu', weights_only=False)
DM_GENES = ckpt['genes']
DM_METS  = ckpt['metabolites']
n_genes = len(DM_GENES)
n_mets  = len(DM_METS)
print(f'genes: {n_genes}, metabolites: {n_mets}')
print(f'sample genes: {DM_GENES[:8]}')
print(f'sample mets:  {DM_METS[:8]}')
print()
print('checkpoint summary:')
for k, v in ckpt['results'].get('summary', {}).items():
    print(f'  {k}: {v}')

# Instantiate the model architecture and load weights
from dark_manifold_100gene import DarkManifold100Gene
model = DarkManifold100Gene(n_genes=n_genes, n_mets=n_mets, hidden_dim=256)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'\nmodel parameters: {sum(p.numel() for p in model.parameters()):,}')

## Cell 3 — knockout sweep

For each of 102 genes, run a knockout via `enzyme_mask[gene_idx] = 0` and roll out a trajectory. Extract two essentiality signals:

* **ATP drop** at terminal step — `atp_t_final − atp_wt_final`
* **Energy charge drop** — `EC_t_final − EC_wt_final` where `EC = (ATP + 0.5·ADP) / (ATP + ADP + AMP)`

A gene is predicted essential if `|atp_drop|` exceeds a threshold. We sweep thresholds in cell 5 to find the optimal MCC.

In [ ]:
import torch
import numpy as np
import time

ROLLOUT_STEPS = 100
ATP_IDX  = DM_METS.index('ATP') if 'ATP' in DM_METS else None
ADP_IDX  = DM_METS.index('ADP') if 'ADP' in DM_METS else None
AMP_IDX  = DM_METS.index('AMP') if 'AMP' in DM_METS else None
print(f'ATP idx: {ATP_IDX}, ADP idx: {ADP_IDX}, AMP idx: {AMP_IDX}')

def initial_state(batch=1):
    """Reference state: enzymes ~ 1.0, metabolites ~ 1.0, ATP/ADP/AMP at physiological ratio."""
    state = torch.ones(batch, n_genes + n_mets) * 1.0
    if ATP_IDX is not None: state[:, n_genes + ATP_IDX] = 3.0
    if ADP_IDX is not None: state[:, n_genes + ADP_IDX] = 1.0
    if AMP_IDX is not None: state[:, n_genes + AMP_IDX] = 0.3
    return state

def rollout(model, state, enzyme_mask, n_steps):
    s = state.clone()
    for _ in range(n_steps):
        new_state = model(s, enzyme_mask=enzyme_mask)
        # The forward returns a state tensor of same shape (per architecture).
        # If the architecture's forward returns a dict, adapt:
        if isinstance(new_state, dict):
            new_state = new_state.get('state', new_state.get('next_state', new_state))
        s = new_state
    return s

# Wild-type baseline
with torch.no_grad():
    wt_mask = torch.ones(1, n_genes)
    s0 = initial_state()
    wt_final = rollout(model, s0, wt_mask, ROLLOUT_STEPS)
    wt_atp = wt_final[0, n_genes + ATP_IDX].item() if ATP_IDX is not None else 0.0
    wt_adp = wt_final[0, n_genes + ADP_IDX].item() if ADP_IDX is not None else 0.0
    wt_amp = wt_final[0, n_genes + AMP_IDX].item() if AMP_IDX is not None else 0.0
    wt_ec = (wt_atp + 0.5 * wt_adp) / max(wt_atp + wt_adp + wt_amp, 1e-9)
    print(f'WT terminal: ATP={wt_atp:.3f}  ADP={wt_adp:.3f}  AMP={wt_amp:.3f}  EC={wt_ec:.3f}')

# Per-gene KO
rows = []
t0 = time.time()
with torch.no_grad():
    for i, gene in enumerate(DM_GENES):
        mask = torch.ones(1, n_genes)
        mask[0, i] = 0.0
        ko_final = rollout(model, initial_state(), mask, ROLLOUT_STEPS)
        atp = ko_final[0, n_genes + ATP_IDX].item() if ATP_IDX is not None else 0.0
        adp = ko_final[0, n_genes + ADP_IDX].item() if ADP_IDX is not None else 0.0
        amp = ko_final[0, n_genes + AMP_IDX].item() if AMP_IDX is not None else 0.0
        ec  = (atp + 0.5 * adp) / max(atp + adp + amp, 1e-9)
        rows.append({
            'gene': gene,
            'ko_atp': atp,
            'ko_ec':  ec,
            'atp_drop':  atp - wt_atp,
            'ec_drop':   ec - wt_ec,
            'abs_atp_drop': abs(atp - wt_atp),
            'abs_ec_drop':  abs(ec - wt_ec),
        })
elapsed = time.time() - t0
print(f'\nknockout sweep: {len(rows)} genes in {elapsed:.1f}s')
import pandas as pd
df_ko = pd.DataFrame(rows)
df_ko.head(8)

## Cell 4 — map Dark Manifold gene names → locus tags (BOTH organisms)

The Dark Manifold uses standard bacterial gene names like `pyk`. Map to:

* `JCVISYN3A_xxxx` via the Syn3A GenBank `/gene` qualifier (Breuer 2019 ground truth).
* `MPN###` via the M. pneumoniae GenBank `/gene` qualifier with `/old_locus_tag` rekey (Lluch-Senar 2015 ground truth).

Both ground truths are in `labels.csv` already (`organism='syn3a'` and `organism='mpne'`).

In [ ]:
from Bio import SeqIO
import pandas as pd, re

# --- Syn3A: gene -> JCVISYN3A_xxxx ---
SYN3A_GB = CELL_REPO / 'cell_sim/data/Minimal_Cell_ComplexFormation/input_data/syn3A.gb'
syn3a_rec = next(SeqIO.parse(str(SYN3A_GB), 'genbank'))
syn3a_name_to_locus = {}
for f in syn3a_rec.features:
    if f.type != 'CDS': continue
    locus = f.qualifiers.get('locus_tag', [None])[0]
    gene  = f.qualifiers.get('gene', [None])[0]
    if locus and gene:
        syn3a_name_to_locus[gene.lower()] = locus
print(f'Syn3A /gene -> locus map: {len(syn3a_name_to_locus)} entries')

# --- M. pneumoniae: gene -> MPN### (preferring legacy /old_locus_tag) ---
MPNE_GB = CELL_REPO / 'memory_bank/data/multiorg_essentiality/raw/mpneumoniae_M129_NC_000912.gb'
mpne_rec = next(SeqIO.parse(str(MPNE_GB), 'genbank'))
mpne_name_to_locus = {}
for f in mpne_rec.features:
    if f.type != 'CDS': continue
    quals = f.qualifiers
    gene = quals.get('gene', [None])[0]
    if not gene: continue
    old = quals.get('old_locus_tag', [None])[0]
    locus = old or quals.get('locus_tag', [None])[0]
    if not locus: continue
    if ',' in str(locus):
        locus = str(locus).split(',')[0].strip()
    if re.match(r'^MPN\d+', str(locus)):
        mpne_name_to_locus[gene.lower()] = locus
print(f'M. pneumoniae /gene -> MPN locus map: {len(mpne_name_to_locus)} entries')

# Map Dark Manifold genes to BOTH organisms
df_ko['gene_lower'] = df_ko['gene'].str.lower()
df_ko['syn3a_locus'] = df_ko['gene_lower'].map(syn3a_name_to_locus)
df_ko['mpne_locus']  = df_ko['gene_lower'].map(mpne_name_to_locus)
n_syn3a = df_ko['syn3a_locus'].notna().sum()
n_mpne  = df_ko['mpne_locus'].notna().sum()
print(f'\nDM 102 genes mapped: syn3a={n_syn3a}, mpne={n_mpne}')
df_ko[['gene', 'syn3a_locus', 'mpne_locus']].head(15)

## Cell 5 — validation against Breuer 2019 + Lluch-Senar 2015

**Important framing:** the Dark Manifold's 102-gene set is curated around central metabolism + tRNA synthetases + ribosomal core + ATP synthase + transcription core. Those gene classes are essential by design in any minimal-cell organism. So when we join to ground-truth essentiality labels (Breuer's Syn3A or Lluch-Senar's M. pneumoniae), the matched subset is dominated by essentials — usually 100% essential.

This makes binary MCC undefined (single-class). Instead, the cell below reports rank-based / recall-at-K metrics that work on single-class subsets:

* **Recall at threshold = 0** (any nonzero atp drop): fraction of matched essentials predicted essential
* **Recall at threshold = median |atp_drop| across all 102 DM genes**: do matched essentials skew above the median?
* **Top-K precision**: of the model's top-K predicted-essential genes (by |atp_drop|), how many of the K are matched essentials?

Plus the original MCC sweep is still computed and reported as undefined when the matched subset is single-class.

In [ ]:
from sklearn.metrics import matthews_corrcoef, confusion_matrix
import pandas as pd, numpy as np

LABELS_CSV = CELL_REPO / 'memory_bank/data/multiorg_essentiality/labels.csv'
labels = pd.read_csv(LABELS_CSV)
print(f'labels.csv: {len(labels)} rows ({labels.organism.value_counts().to_dict()})')

results = {}

def validate_one_organism(org_key, locus_col, label_source):
    """Compute recall-at-K + MCC sweep for one organism's labels."""
    org_labels = labels[labels.organism == org_key][['locus_tag', 'essential']].rename(
        columns={'essential': 'gt_essential'})
    print(f'\n=== {label_source} ({org_key}) ===')
    print(f'  total {org_key} labels: {len(org_labels)} '
          f'({int(org_labels.gt_essential.sum())} essential, '
          f'{int((1-org_labels.gt_essential).sum())} nonessential)')

    joined = df_ko.merge(org_labels, left_on=locus_col, right_on='locus_tag', how='inner')
    n = len(joined)
    n_ess = int(joined.gt_essential.sum())
    n_non = int(n - n_ess)
    print(f'  DM matched + with {org_key} label: {n} ({n_ess} essential, {n_non} nonessential)')
    if n == 0:
        return {'organism': org_key, 'mapped': 0, 'mcc': None,
                'mcc_status': 'no_genes_mapped'}

    # ---- recall@thresholds ----
    median_drop = float(df_ko.abs_atp_drop.median())
    recall_at_zero = float((joined.abs_atp_drop > 0).mean()) if n_ess > 0 else None
    recall_at_median = float((joined.abs_atp_drop >= median_drop).mean()) if n_ess > 0 else None

    # ---- top-K precision (does the model rank essentials above the cohort median?) ----
    df_ko_sorted = df_ko.sort_values('abs_atp_drop', ascending=False)
    K = n_ess
    top_K_genes = set(df_ko_sorted.head(K)['gene'].tolist())
    matched_essential_genes = set(joined[joined.gt_essential == 1]['gene'].tolist())
    top_K_in_matched_essential = len(top_K_genes & matched_essential_genes)
    top_K_precision = top_K_in_matched_essential / K if K > 0 else None

    print(f'  recall @ |atp_drop|>0:           {recall_at_zero}')
    print(f'  recall @ |atp_drop|>={median_drop:.4f} (median): {recall_at_median}')
    print(f'  top-{K} predicted-essential overlap: {top_K_in_matched_essential}/{K} = {top_K_precision}')

    # ---- MCC sweep (will be undefined if single-class; reported as null) ----
    if n_ess > 0 and n_non > 0:
        thresholds = sorted(set(np.linspace(0, joined.abs_atp_drop.max() * 1.01, 200)) |
                             set(joined.abs_atp_drop.tolist()))
        mcc_results = []
        for thr in thresholds:
            pred = (joined.abs_atp_drop >= thr).astype(int)
            if pred.sum() == 0 or pred.sum() == len(pred):
                mcc = 0.0
            else:
                mcc = matthews_corrcoef(joined.gt_essential, pred)
            mcc_results.append({'threshold': thr, 'n_pred_essential': int(pred.sum()), 'mcc': mcc})
        df_mcc = pd.DataFrame(mcc_results).sort_values('mcc', ascending=False)
        best = df_mcc.iloc[0]
        best_pred = (joined.abs_atp_drop >= best.threshold).astype(int)
        cm = confusion_matrix(joined.gt_essential, best_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        mcc_status = 'measured'
        mcc_value = float(best.mcc)
        mcc_threshold = float(best.threshold)
        print(f'  best MCC threshold: {best.threshold:.4f}')
        print(f'  best MCC: {best.mcc:.4f}')
        print(f'  TP={tp} FP={fp} TN={tn} FN={fn}')
    else:
        print(f'  MCC: undefined (matched subset has only one class)')
        mcc_status = 'undetermined_single_class_subset'
        mcc_value = None
        mcc_threshold = None
        tp = n_ess; fp = 0; tn = 0; fn = 0

    return {
        'organism': org_key,
        'label_source': label_source,
        'mapped': n,
        'matched_essential': n_ess,
        'matched_nonessential': n_non,
        'recall_at_zero': recall_at_zero,
        'recall_at_median_atp_drop': recall_at_median,
        'median_atp_drop_across_dm_genes': median_drop,
        'top_K_precision': top_K_precision,
        'top_K_overlap': top_K_in_matched_essential,
        'top_K': K,
        'mcc': mcc_value,
        'mcc_status': mcc_status,
        'mcc_threshold': mcc_threshold,
        'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
    }

results['syn3a'] = validate_one_organism('syn3a', 'syn3a_locus', 'Breuer 2019 eLife')
results['mpne']  = validate_one_organism('mpne',  'mpne_locus',  'Lluch-Senar 2015 MSB')
print('\n\n=== summary ===')
for k, v in results.items():
    print(f'  {k}: mapped={v["mapped"]}  '
          f'mcc_status={v["mcc_status"]}  '
          f'recall@0={v.get("recall_at_zero")}  '
          f'recall@median={v.get("recall_at_median_atp_drop")}  '
          f'topK={v.get("top_K_overlap")}/{v.get("top_K")}')

## Cell 6 — write results to outputs/dark_manifold_validation.json

Both organisms' results are dumped together so the downstream fact captures the full picture.

In [ ]:
import json

out = {
    'method': 'dark_manifold_100gene_knockout_sweep_dual_organism',
    'rollout_steps': ROLLOUT_STEPS,
    'n_dm_genes': int(n_genes),
    'n_dm_mets':  int(n_mets),
    'n_dm_genes_mapped_syn3a': int(n_syn3a),
    'n_dm_genes_mapped_mpne':  int(n_mpne),
    'organisms': results,
    'baselines': {
        'v15_syn3a_biology_first': 0.5372,
        'v18_xgboost_syn3a_held_out': 0.1376,
    },
    'caveats': [
        'Dark Manifold 102-gene set is curated around central metabolism + tRNA synthetases + ribosomal core + ATP synthase + transcription core, which are essential by design in any minimal-cell organism. Joining to ground-truth essentiality on either Syn3A or M. pneumoniae produces a single-class matched subset (100 percent essential), so binary MCC is mathematically undefined.',
        'Recall-at-K and top-K precision are reported as alternative metrics that work on single-class subsets. They measure whether the model ranks matched essentials above or near the cohort median.',
        'Dark Manifold trained on FBA-synthetic ground truth (cell-simulation/dark_manifold_100gene.py MetabolicGroundTruth class), not measured biology. The internal 0.95+ gene correlation is correlation against the FBA model, not against measured experimental data.',
        'cell-simulation/GAP_ANALYSIS.md flags hub-gene wrong-sign predictions (groEL, rpoB, dnaA all wrong direction) and W_reg learned to be 100 percent zeros. Those known issues are present in the loaded checkpoint.',
    ],
}
out_path = CELL_REPO / 'outputs/dark_manifold_validation.json'
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'wrote {out_path}\n')
print(json.dumps(out, indent=2))

## Cell 7 — push results back to the branch (optional)

If you set `GITHUB_TOKEN` in Colab secrets (or as an env var), this commits the JSON output back to the dev branch.

In [ ]:
import os, subprocess

GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')
if not GITHUB_TOKEN:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = ''
if not GITHUB_TOKEN:
    print('Skip: set GITHUB_TOKEN in Colab Secrets (left sidebar key icon) or as env var to push results back.')
else:
    os.chdir(CELL_REPO)
    subprocess.run(['git', 'config', 'user.email', 'colab@noreply.local'], check=True)
    subprocess.run(['git', 'config', 'user.name',  'Colab Validation'], check=True)
    subprocess.run(['git', 'remote', 'set-url', 'origin',
                    f'https://{GITHUB_TOKEN}@github.com/Nikku03/cell.git'], check=True)
    subprocess.run(['git', 'add', 'outputs/dark_manifold_validation.json'], check=True)
    subprocess.run(['git', 'commit', '-m',
                    'data: Dark Manifold v20b validation against Breuer + Lluch-Senar (Colab notebook output)'],
                   check=True)
    subprocess.run(['git', 'push', 'origin',
                    'claude/syn3a-whole-cell-simulator-REjHC'], check=True)
    print('Pushed to claude/syn3a-whole-cell-simulator-REjHC')